<a href="https://colab.research.google.com/github/zhaoyingjun/Tiny-R2/blob/main/Tiny_R2_journey.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tiktoken datasets transformers huggingface_hub
!pip install git+https://github.com/KellerJordan/Muon
!pip install --upgrade transformers


In [ ]:
!pip install faiss-cpu sentence_transformers datasets rouge-score nltk rank_bm25 tqdm

In [ ]:
!hf auth login --force

In [3]:
!git clone https://github.com/zhaoyingjun/Tiny-R2.git
%cd Tiny-R2

Cloning into 'Tiny-R2'...
remote: Enumerating objects: 463, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 463 (delta 12), reused 9 (delta 9), pack-reused 445 (from 1)
Receiving objects: 100% (463/463), 3.94 MiB | 34.49 MiB/s, done.
Resolving deltas: 100% (271/271), done.
/content/Tiny-R2


**训练climbmix数据，可以训练一个基础大模型，1.7B参数量，评测模型结构和效果。支持采用大模型作为Agent进行观察训练过程并给出实施的lr、clip的调整参数;--use_agent_observe=True开启,默认是关闭的，需要输入gemini的api_key**

默认不开启Agent智能化自主训练

In [ ]:
!python train.py --n_layer 6 --n_embd 1536 --hc 'True' --mhc 'True' --n_experts 8 --max_iters 10000 --attention_types 'Sparse' --batch_size 8 --ctx_len 2048 --hf_dataset 'karpathy/climbmix-400b-shuffle' --resume True --save_best_only True

开启Agent智能化自主训练，需要将your gemini api key替换

In [ ]:
!python train.py --n_layer 6 --n_embd 1536 --hc 'True' --mhc 'True' --n_experts 8 --max_iters 10000 --attention_types 'Sparse' --batch_size 8 --ctx_len 2048 --hf_dataset 'karpathy/climbmix-400b-shuffle' --resume True --save_best_only True --use_agent_observe True --gemini_api_key 'your gemini api key'

预训练完成之后，进行benchmark的验证

In [ ]:
!python /content/Tiny-R2/evaluate.py --checkpoint /content/Tiny-R2/checkpoints/best_model_step_1180.pt

后训练直接使用Qwen3.5-9B模型作为教师模型进行OPD训练，可自定义使用RAG增加教师模型以及自定义数据集等

In [ ]:
!python opd_train.py --hf_teacher_model Qwen/Qwen3.5-9B \
  --student_model_name tiny-r2 \
  --student_ckpt checkpoints/best_model_step_40.pt \
  --batch_size 2 \
  --grad_accum_steps 4 \
  --custom_qa_path baoxianqa.jsonl \
  --rag_corpus_path baoxianqa.txt

In [ ]:
!pip install flash-linear-attention causal-conv1d

使用Qwen3.5-9B模型作为教师模型、Qwen3.5-0.8B作为学生模型进行OPD训练，用RAG增加教师模型，RAG数据集来自问答数据集集medquad，可以复现Readme中的结果;去除命令行中--enable_reg_teacher则关闭外挂RAG功能；--rag_corpus_path外挂RAG数据集，--custom_qa_path 自定义问题集

In [ ]:
!python opd_train.py \
  --hf_teacher_model Qwen/Qwen3.5-9B \
  --student_model_name Qwen/Qwen3.5-0.8B \
  --batch_size 2 \
  --grad_accum_steps 4

✅ 成功导入 rank_bm25 与 sentence_transformers，混合 RAG 模块已启用。
✅ 成功导入 Tiny-R2 核心模块
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
💡 [System] 初始化 Gloo 分布式环境以保持优化器兼容。

🚀 启动学术重构版 Agent OPD 蒸馏训练
👨‍🏫 教师模型: Qwen/Qwen3.5-9B
👶 学生模型: Qwen/Qwen3.5-0.8B (Ablation Mode: none)
🎯 学生端 RAG  : 禁用 - 完全封闭知识内化

📡 载入 Hugging Face 线上数据集: pubmed_qa
🔒 正在构建标准 RAG 语料库...
[-] 检索语料库构建完成，共包含 1000 条文档。
[-] 正在初始化过滤后的 BM25 词法索引...
[-] 正在初始化密向量检索模型 (BAAI/bge-small-en-v1.5)...
Loading weights: 100% 199/199 [00:00<00:00, 4248.40it/s]
Batches: 100% 32/32 [00:01<00:00, 17.68it/s]
📡 尝试预载 OOD 评测数据集 (MedQA)...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100% 320/320 [00:00<00:00, 816.84it/s]

默认使用medquade数据集进行OPD的结果验证，可以根据OPD训练的数据集对应的修改验证

In [ ]:
!python eval_opd.py \
    --dataset medquad \
    --hf_teacher_model Qwen/Qwen3.5-9B \
    --student_base_model Qwen/Qwen3.5-0.8B \
    --student_opd_model opd_checkpoints/student_best_model_step_100.pt \
    --eval_samples 10